# Predicción de Churn en Telco — 04 · Aplicación del modelo final

Carga el modelo generado por `03_entrenar_modelo_final.ipynb` y lo aplica a los datos
de test (el 20% reservado, nunca usado durante el desarrollo). Al ejecutarse de punta a
punta replica exactamente el archivo de predicciones `predicciones_test.csv`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42

def cargar_datos():
    """Carga y limpieza minima definida en 01_eda.ipynb."""
    df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
    X = df.drop(columns=["customerID", "Churn"])
    y = (df["Churn"] == "Yes").astype(int)
    ids = df["customerID"]
    return X, y, ids

def particion(X, y, ids):
    """Misma particion estratificada 80/20 usada en 02_experimentos.ipynb."""
    return train_test_split(X, y, ids, test_size=0.20, stratify=y, random_state=SEED)

In [2]:
import json
import joblib

X, y, ids = cargar_datos()
X_train, X_test, y_train, y_test, ids_train, ids_test = particion(X, y, ids)

modelo = joblib.load("modelo_final.joblib")
with open("umbral_operativo.json") as f:
    umbral = json.load(f)["umbral"]

score = modelo.predict_proba(X_test)[:, 1]
predicciones = pd.DataFrame({
    "customerID": ids_test.values,
    "score_churn": score,
    "pred_churn": (score >= umbral).astype(int),
})
predicciones.to_csv("predicciones_test.csv", index=False)
print(f"{len(predicciones)} predicciones -> predicciones_test.csv "
      f"(umbral {umbral:.4f}, {predicciones.pred_churn.mean():.1%} marcados en riesgo)")
predicciones.head()

1409 predicciones -> predicciones_test.csv (umbral 0.1526, 52.5% marcados en riesgo)


C:\Users\ftamaki\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,customerID,score_churn,pred_churn
0,4376-KFVRS,0.048802,0
1,2754-SDJRD,0.747721,1
2,9917-KWRBE,0.061089,0
3,0365-GXEZS,0.287827,1
4,9385-NXKDA,0.029586,0
